## Goal

The objective of this notebook is to design the feature preprocessing strategy required for machine learning models.

The preprocessing workflow focuses on preparing numerical and categorical features for model training by selecting appropriate encoding and scaling techniques while preventing data leakage.

Rather than permanently modifying the dataset, preprocessing decisions are documented and validated so they can later be integrated into a reproducible Scikit-learn Pipeline during model development.

## Preprocessing Tasks

Based on the exploratory data analysis and data cleaning stages, the following preprocessing tasks have been identified:

- Identify numerical and categorical features.
- Select an appropriate encoding strategy for categorical variables.
- Select an appropriate scaling technique for numerical variables.
- Design a preprocessing workflow compatible with Scikit-learn Pipelines.
- Prepare the dataset for future model training while avoiding data leakage.

## Categorical Features

Most categorical variables require numerical encoding before they can be used by machine learning algorithms.

Binary categorical features (e.g., Yes/No) and multi-class categorical features (e.g., InternetService, Contract, PaymentMethod) will require different encoding strategies depending on their characteristics.

The appropriate encoding methods will be selected during the implementation phase.

## Numerical Features

Numerical features are measured on different scales.

For example, TotalCharges contains considerably larger values than tenure or MonthlyCharges.

Scaling these features helps prevent machine learning algorithms from giving disproportionate importance to variables solely because of their numerical magnitude.

The final scaling strategy will be selected during model development.

## Engineering Decision

Feature preprocessing will not be permanently applied to the dataset at this stage.

Instead, preprocessing operations will later be incorporated into a Scikit-learn Pipeline together with the machine learning model.

This approach guarantees that all preprocessing steps are learned exclusively from the training data, preventing data leakage and ensuring a reproducible workflow.

In [1]:
# Imports
import numpy as np 
import pandas as pd

In [2]:
clean_df = pd.read_csv('../data/processed/customer_churn_clean.csv')
clean_df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7028,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7029,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7030,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [3]:
# Dataset overview
clean_df.shape

(7032, 20)

In [4]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   str    
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   str    
 3   Dependents        7032 non-null   str    
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   str    
 6   MultipleLines     7032 non-null   str    
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   str    
 9   OnlineBackup      7032 non-null   str    
 10  DeviceProtection  7032 non-null   str    
 11  TechSupport       7032 non-null   str    
 12  StreamingTV       7032 non-null   str    
 13  StreamingMovies   7032 non-null   str    
 14  Contract          7032 non-null   str    
 15  PaperlessBilling  7032 non-null   str    
 16  PaymentMethod     7032 non-null   str    
 17  Monthl

### Feature Classification

| Feature        | Data Type | ML Type            | Needs Encoding | Needs Scaling  |
| -------------- | --------- | ------------------ | -------------- | -------------- |
| gender         | object    | Binary Categorical | ✅              | ❌              |
| Partner        | object    | Binary Categorical | ✅              | ❌              |
| Dependents     | object    | Binary Categorical | ✅              | ❌              |
| PhoneService   | object    | Binary Categorical | ✅              | ❌              |
| MultipleLines  | object    | Multi-class Categorical | ✅              | ❌              |
| InternetService| object    | Multi-class Categorical | ✅              | ❌              |
| OnlineSecurity | object    | Multi-class Categorical | ✅              | ❌              |
| OnlineBackup   | object    | Multi-class Categorical | ✅              | ❌              |
| DeviceProtection| object    | Multi-class Categorical | ✅              | ❌              |
| TechSupport    | object    | Multi-class Categorical | ✅              | ❌              |
| StreamingTV    | object    | Multi-class Categorical | ✅              | ❌              |
| StreamingMovies| object    | Multi-class Categorical | ✅              | ❌              |
| PaperlessBilling| object    | Binary Categorical | ✅              | ❌              |
| PaymentMethod  | object    | Multi-class Categorical | ✅              | ❌              |
| Contract       | object    | Multi-class        | ✅              | ❌              |
| MonthlyCharges | float     | Numerical          | ❌              | ✅              |
| TotalCharges   | float     | Numerical          | ❌              | ✅              |
| tenure         | int       | Numerical          | ❌              | ✅              |
| SeniorCitizen  | int       | Binary             | ❌              | ❌ (current decision) |


### Encoding Strategy

Binary Features

↓

Using .map() cause it is more readable, convinient, easy to implement in pipeline and understandable.

Multi-class Features

↓

Multi-class categorical features will most likely be encoded using One-Hot Encoding because these categories do not possess any natural ordering.
Ordinal Encoding should only be considered when the categories have a meaningful ordinal relationship.

### Scaling Strategy

tenure

MonthlyCharges

TotalCharges

---

Numerical features will be standardized using StandardScaler.

Standardization transforms numerical variables to have approximately zero mean and unit variance, preventing features with larger numerical magnitudes from dominating the learning process.

Binary numerical features such as SeniorCitizen will remain unchanged because they already represent meaningful binary values.

### Preprocessing Plan
```
Categorical Features
        ↓
Encoding
        ↓
Machine Learning Pipeline

Numerical Features
        ↓
Scaling
        ↓
Machine Learning Pipeline
```

In [5]:
# Seperating categorical and numerical features
categorical_features = clean_df.select_dtypes("object").columns.tolist()
numerical_features = clean_df.select_dtypes("number").columns.tolist()

print(categorical_features)
print(numerical_features)

['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


C:\Users\User\AppData\Local\Temp\ipykernel_2064\2645903535.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = clean_df.select_dtypes("object").columns.tolist()


In [7]:
# Creating a feature summary for better categorizing binary & multi-class categorical features & better insight on numerical features
feature_summary = pd.DataFrame({
    "Feature": clean_df.columns,
    "Unique Values": [clean_df[c].nunique() for c in clean_df.columns]
})

feature_summary

,Feature,Unique Values
0,gender,2
1,SeniorCitizen,2
2,Partner,2
3,Dependents,2
4,tenure,72
5,PhoneService,2
6,MultipleLines,3
7,InternetService,3
8,OnlineSecurity,3
9,OnlineBackup,3


In [8]:
# Checking the scales of features
clean_df[numerical_features].agg(["mean", "std", "min", "max"])

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
mean,0.162400,32.421786,64.798208,2283.300441
std,0.368844,24.545260,30.085974,2266.771362
min,0.000000,1.000000,18.250000,18.800000
max,1.000000,72.000000,118.750000,8684.800000


In [9]:
binary_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

multiclass_features = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

passthrough_features = [
    "SeniorCitizen"
]

In [10]:
preprocessing_plan = {
    "Binary": binary_features,
    "MultiClass": multiclass_features,
    "Numerical": numerical_features,
    "Passthrough": passthrough_features
}

preprocessing_plan

{'Binary': ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'PaperlessBilling'],
 'MultiClass': ['MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaymentMethod'],
 'Numerical': ['tenure', 'MonthlyCharges', 'TotalCharges'],
 'Passthrough': ['SeniorCitizen']}

### Engineering Decisions

- Raw data remains unchanged.
- Clean dataset is used as preprocessing input.
- Encoding will be performed inside a Scikit-learn Pipeline.
- Scaling will be fitted only on the training set.
- No permanently encoded dataset will be created.
- SeniorCitizen is already represented as a meaningful binary numerical feature (0/1).
- Since its values are categorical by nature rather than continuous measurements, no scaling will be applied.
- Keeping the original representation preserves interpretability without negatively affecting model performance.

## Summary

The preprocessing requirements have been identified and documented.

Categorical and numerical features have been classified according to their preprocessing needs, and a reproducible preprocessing strategy has been designed.

The actual implementation of encoding and scaling will be performed inside a Scikit-learn Pipeline during the model development stage.

## Sprint Retrospective

### Completed

- Identified categorical and numerical features.
- Designed the preprocessing workflow.
- Documented preprocessing decisions.
- Planned a Pipeline-based preprocessing strategy.

### Key Decisions

- Separate preprocessing from data cleaning.
- Avoid permanent preprocessing of the dataset.
- Integrate preprocessing into the machine learning Pipeline.
- Prevent data leakage by fitting preprocessing only on training data.

### Next Sprint

Build the first baseline classification model using a complete preprocessing Pipeline.